# HS4002 Week 8

## Practising with Count Models

Research question: **what factors influence the number of children someone has?**

We'll compare three model types for count outcomes:
1. OLS (baseline)
2. Poisson regression
3. Negative Binomial regression

Plus: overdispersion testing and marginal effects.

In [ ]:
# !pip install pandas numpy plotnine statsmodels stargazer marginaleffects scipy

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
from scipy.stats import chi2

In [ ]:
raw_df = pd.read_csv("gss2022_mini2.csv")

chosen_variables = [
	'id', 'age', 'happy', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat',
	'income16', 'prestg10', 'degree', 'race', 'sex', 'wrkstat', 'biokids', 'educ'
]
df = raw_df[chosen_variables].dropna().copy()

income_map = {
	1: 500,    2: 2000,   3: 3500,   4: 4500,   5: 5500,   6: 6500,
	7: 7500,   8: 9000,   9: 11250,  10: 13750, 11: 16250, 12: 18750,
	13: 21250, 14: 23750, 15: 27500, 16: 32500, 17: 37500, 18: 45000,
	19: 55000, 20: 67500, 21: 82500, 22: 100000, 23: 120000, 24: 140000,
	25: 160000, 26: 250000
}
df['income_cont']   = df['income16'].map(income_map)
df['income_10k']   = df['income_cont'] / 10000
df['age']       = raw_df.loc[df.index, 'age'] - 18

df['binary_lvmus']  = (df['yrlvmus']  == 1).astype(int)
df['binary_artxbt'] = (df['yrartxbt'] == 1).astype(int)
df['binary_movie']  = (df['yrmovie']  == 1).astype(int)
df['binary_creat']  = (df['yrcreat']  == 1).astype(int)
df['omni'] = df[['binary_lvmus','binary_artxbt','binary_movie','binary_creat']].sum(axis=1)

df['race_bin']   = pd.Categorical(df['race'].astype(str))
df['race_cat']   = df['race'].map({1: 'White', 2: 'Black'}).fillna('Other')
df['ba_binary']  = (df['degree'] >= 3).astype(int)
df['sex_woman']  = pd.Categorical(
    np.where(df['sex'] == 2, 'Woman', 'Not Woman'),
    categories=['Not Woman', 'Woman']
)
df['working']    = np.where(df['wrkstat'] <= 3, 'Working', 'Not Working')
df['happy_bin']  = (df['happy'] <= 2).astype(int)
df['happy_cat']  = np.where(df['happy'] <= 2, 'Happy', 'Not Happy')

# Starting with an OLS model

Regress no. of children (`biokids`) against income in ten thousands (`income_10k`). Note: one unit = $10,000. Name this `ols_lm1`.

In [ ]:
ols_lm1 = smf.ols('biokids ~ income_10k', data=df).fit()
print(ols_lm1.summary())

How would you interpret the coefficient for `income_10k`? What is unusual about using OLS on a count variable?

# Continuing with a Poisson Model

Use `smf.poisson()` — equivalent to `glm(..., family='poisson')` in R.

In [ ]:
po_lm1 = smf.poisson('biokids ~ income_10k', data=df).fit()
print(po_lm1.summary())

Interpret the coefficient on `income_10k`. Do not read it as you would an OLS coefficient.

Poisson coefficients are **factors of change**: a one-unit change in `income_10k` is expected to change the number of children by a factor of exp(β), ceteris paribus.

You can alternatively read this as a **percentage change**: 100 × [exp(β) − 1]%.

In [ ]:
import numpy as np
b = po_lm1.params['income_10k']
print(f'Factor of change, exp(b): {np.exp(b):.4f}')
print(f'Percentage change, 100 * [exp(b) - 1]: {100 * (np.exp(b) - 1):.2f}%')

# Adding Age

Add `age` to both models. `age` is centered at 18, so 0 is an 18-year-old. Save as `ols_lm2` and `po_lm2`.

In [ ]:
ols_lm2 = smf.ols('biokids ~ income_10k + age', data=df).fit()
po_lm2  = smf.poisson('biokids ~ income_10k + age', data=df).fit()
print(po_lm2.summary())

## Comparing the Poisson Models via Likelihood Ratio Test

In [ ]:
lr_stat = 2 * (po_lm2.llf - po_lm1.llf)
df_diff = po_lm2.df_model - po_lm1.df_model
p_val   = chi2.sf(lr_stat, df_diff)
print(f'LR statistic = {lr_stat:.4f}, df = {df_diff:.0f}, p-value = {p_val:.6f}')

# Building a Negative Binomial Model

Negative binomial relaxes the Poisson assumption that variance = mean (allowing overdispersion).

Equivalent to R's `MASS::glm.nb()`.

Read the coefficients the same way as in a Poisson model: as a factor of change, exp(β), or as a percentage change, 100 × [exp(β) − 1]%.

In [ ]:
nb_m1 = smf.negativebinomial('biokids ~ income_10k + age', data=df).fit()
print(nb_m1.summary())

In [ ]:
from stargazer.stargazer import Stargazer

sg = Stargazer([ols_lm2, po_lm2, nb_m1])
sg.title('Number of Children')
sg.custom_columns(['OLS', 'Poisson', 'Neg. Binomial'], [1, 1, 1])

# Ending the cell on the object itself renders the table in the notebook
sg

## Test for Overdispersion

Was NB necessary? We test whether the NB's dispersion parameter (alpha) is significantly different from 0. If alpha = 0, the NB reduces to Poisson.

This is equivalent to R's `pscl::odTest()`.

In [ ]:
# Overdispersion test: LR comparing NB vs Poisson
# Under H0 (Poisson), the test statistic has a chi-square(1)/2 distribution
# because the null is on the boundary of the parameter space
lr_od = 2 * (nb_m1.llf - po_lm2.llf)
p_od  = 0.5 * chi2.sf(lr_od, 1)
print(f'Overdispersion LR statistic = {lr_od:.4f}')
print(f'p-value (one-sided) = {p_od:.6f}')
print('NB preferred over Poisson:', p_od < 0.05)

# Marginal Effects

A marginal effect is the expected change in the number of children when an independent variable increases by one unit, ceteris paribus. Compute them the same way as for the logit models.

In [ ]:
from marginaleffects import avg_comparisons, comparisons

## Average marginal effects of income

In [ ]:
avg_comparisons(po_lm2, variables='income_10k')

In [ ]:
avg_comparisons(nb_m1, variables='income_10k')

## Marginal effect at the mean of income

In [ ]:
comparisons(po_lm2, variables='income_10k', newdata='mean')

In [ ]:
comparisons(nb_m1, variables='income_10k', newdata='mean')

# Independent Investigation: Education and Number of Children

Investigate the relationship between **no. of children** (`biokids`) and **BA education** (`ba_binary`).

Do the following:

1. Include these independent variables: `ba_binary`, `age`, `income_10k`, `sex_woman`, and `race_bin`.
2. Build OLS, Poisson, and Negative Binomial models.
3. Produce a Stargazer table comparing the three models.
4. Compute marginal effects for each model.
5. Decide which model is best. Justify.
6. Answer: does a person with a BA have more or fewer children, and by how much? For the Poisson and negative binomial models, report a factor of change or a percentage change.